# 02. Exploratory Data Analysis (EDA)

**Wejście:** `fuel_segments_train` (analiza rozwojowa bez podglądania zbioru testowego). Danych `rank` używamy później do walidacji, a `final` wyłącznie do końcowej oceny modelu.



In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

CATALOG = "pk_big_data"
SCHEMA = "curated"

def table(name):
    return spark.table(f"{CATALOG}.{SCHEMA}.{name}")

def show(df, n=20):
    display(df.limit(n))

train = table("fuel_segments_train")
print("Wiersze:", train.count(), "Kolumny:", len(train.columns))
train.printSchema()
show(train)

Wiersze: 131530 Kolumny: 75
root
 |-- flight_id: string (nullable = true)
 |-- idx: long (nullable = true)
 |-- start: long (nullable = true)
 |-- end: long (nullable = true)
 |-- fuel_kg: double (nullable = true)
 |-- start_converted: timestamp (nullable = true)
 |-- end_converted: timestamp (nullable = true)
 |-- alt_mean: double (nullable = true)
 |-- alt_min: double (nullable = true)
 |-- alt_max: double (nullable = true)
 |-- alt_std: double (nullable = true)
 |-- gs_mean: double (nullable = true)
 |-- gs_min: double (nullable = true)
 |-- gs_max: double (nullable = true)
 |-- gs_std: double (nullable = true)
 |-- vr_mean: double (nullable = true)
 |-- vr_min: double (nullable = true)
 |-- vr_max: double (nullable = true)
 |-- vr_std: double (nullable = true)
 |-- mach_mean: double (nullable = true)
 |-- mach_min: double (nullable = true)
 |-- mach_max: double (nullable = true)
 |-- mach_std: double (nullable = true)
 |-- tas_mean: double (nullable = true)
 |-- tas_min: double (nu

flight_id,idx,start,end,fuel_kg,start_converted,end_converted,alt_mean,alt_min,alt_max,alt_std,gs_mean,gs_min,gs_max,gs_std,vr_mean,vr_min,vr_max,vr_std,mach_mean,mach_min,mach_max,mach_std,tas_mean,tas_min,tas_max,tas_std,cas_mean,cas_min,cas_max,cas_std,lat_mean,lat_min,lat_max,lat_std,lon_mean,lon_min,lon_max,lon_std,track_mean,track_min,track_max,track_std,n_points,n_acars,n_adsb,first_point_ns,last_point_ns,alt_first,alt_last,lat_first,lat_last,lon_first,lon_last,dur_s,t_from_first_s,alt_change,flight_flight_date,flight_aircraft_type,flight_takeoff,flight_landed,flight_origin_icao,flight_origin_name,flight_destination_icao,flight_destination_name,origin_airport_icao,origin_airport_longitude,origin_airport_latitude,origin_airport_elevation,destination_airport_icao,destination_airport_longitude,destination_airport_latitude,destination_airport_elevation,route_km,co2_kg
prc770822360,0,1744511464447000000,1744513264487000000,2500.0,2025-04-13T02:31:04.447Z,2025-04-13T03:01:04.487Z,35988.58500078253,35974.9344,35999.928,12.44544527391049,472.013723150358,465.0,480.0,4.0544565027709,-3.360381861575179,-64.0,64.0,46.37252357699412,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,46.237736875519644,45.18333333333333,47.22285461425781,0.6103391099430924,21.883091492274225,19.505882263183594,24.35,1.4556952989518255,301.22856397293026,300.1515479818221,302.29205517325994,0.560419024596875,3077,1,3076,1744511464447000000,1744513264184999943,null,35999.928,45.18333333333333,47.22285461425781,24.35,19.505882263183594,1800.04,0.0,null,2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,WMKK,101.709999,2.74558,69,EHAM,4.76389,52.308601,-11,10236.638941091825,7900.0
prc770822360,2,1744514164458000000,1744515964417000000,2500.0,2025-04-13T03:16:04.458Z,2025-04-13T03:46:04.417Z,35988.99060517793,35974.9344,35999.928,12.400906479713907,511.88579387186627,493.0,525.0,9.377808426969006,-36.040854224698236,-128.0,64.0,43.789453207161394,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,49.48169800519137,48.233333333333334,50.713531494140625,0.7165572021706532,14.338007378866664,11.638307313661318,16.95,1.5286036074014282,305.6584113839573,303.9986040132782,307.34229030628615,0.7405114823611789,3479,1,3478,1744514164458000000,1744515963571000099,null,35999.928,48.233333333333334,50.713531494140625,16.95,11.638307313661318,1799.959,2700.011,null,2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,WMKK,101.709999,2.74558,69,EHAM,4.76389,52.308601,-11,10236.638941091825,7900.0
prc770822360,1,1744513264487000000,1744514164226000000,1200.0000000000011,2025-04-13T03:01:04.487Z,2025-04-13T03:16:04.226Z,35989.29034362479,35974.9344,35999.928,12.361191419446147,484.7035490605428,475.0,497.0,7.434830617053677,-29.862212943632567,-64.0,64.0,36.801336411330745,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,47.7217855326957,47.21666666666667,48.24165344238281,0.28755524028346957,18.226099218672655,16.94908728966346,19.516666666666666,0.7411761469703806,300.68957777902773,298.80569035120664,306.6150454751361,2.521644394838623,1789,1,1788,1744513264487000000,1744514163982000113,null,35974.9344,47.21666666666667,48.24165344238281,19.516666666666666,16.94908728966346,899.739,1800.04,null,2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,WMKK,101.709999,2.74558,69,EHAM,4.76389,52.308601,-11,10236.638941091825,3792.0000000000036
prc770822360,3,1744515964417000000,1744516864540000000,1299.9999999999998,2025-04-13T03:46:04.417Z,2025-04-13T04:01:04.540Z,35988.4749830669,35949.940800000004,36024.9216,13.181922965468585,506.3889816360601,499.0,515.0,4.1066336454252585,-34.45742904841402,-256.0,128.0,64.97460032545939,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,51.2429791

## 1. Sprawdzenie danych i unikalności przedziałów

In [0]:
assert {"idx", "flight_id", "fuel_kg"}.issubset(train.columns)
print("Duplikaty idx:", train.groupBy("idx").count().filter("count > 1").count())
print("Braki ID:", train.filter(F.col("idx").isNull() | F.col("flight_id").isNull()).count())
print("Braki paliwa:", train.filter(F.col("fuel_kg").isNull() | F.isnan("fuel_kg")).count())
print("Ujemne paliwo:", train.filter(F.col("fuel_kg") < 0).count())
if "dur_s" in train.columns:
    show(train.select(F.min("dur_s").alias("min_s"), F.max("dur_s").alias("max_s")))

Duplikaty idx: 0
Braki ID: 0
Braki paliwa: 0
Ujemne paliwo: 0


min_s,max_s
1.61,3599.993


## 2. Braki danych w każdej kolumnie
Wyniki pomagają wybrać później strategię imputacji. `NaN` sprawdzamy tylko w kolumnach numerycznych.

In [0]:
from pyspark.sql.types import NumericType
N = train.count()
missing_exprs = []
for f in train.schema.fields:
    c = F.col(f.name)
    test = c.isNull()
    if isinstance(f.dataType, NumericType):
        test = test | F.isnan(c.cast("double"))
    missing_exprs.append(F.sum(F.when(test, 1).otherwise(0)).alias(f.name))
missing = train.agg(*missing_exprs).first().asDict()
missing_df = spark.createDataFrame(
    [(k, int(v or 0), round(100 * (v or 0) / N, 2) if N else None) for k,v in missing.items()],
    ["column", "missing_n", "missing_pct"]
)
show(missing_df.orderBy(F.desc("missing_pct")), 100)

column,missing_n,missing_pct
mach_std,131530,100.0
tas_std,131530,100.0
cas_std,131530,100.0
cas_mean,110180,83.77
cas_min,110180,83.77
cas_max,110180,83.77
tas_mean,87077,66.2
tas_min,87077,66.2
tas_max,87077,66.2
mach_mean,47314,35.97


## 3. Rozkład zużycia paliwa i emisji CO₂
`co2_kg` jest wyliczone z `fuel_kg` — nie używamy go jako predyktora paliwa.

In [0]:
cols = [c for c in ["fuel_kg", "co2_kg", "dur_s", "gs_mean", "alt_mean", "route_km", "n_points"] if c in train.columns]
show(train.select(*cols).describe())
show(train.select(
    F.expr("percentile_approx(fuel_kg, array(0.01, 0.25, 0.50, 0.75, 0.99), 10000)").alias("fuel_quantiles_kg")
))

summary,fuel_kg,co2_kg,dur_s,gs_mean,alt_mean,route_km,n_points
count,131530,131530,131530,85390,127358,131530,131530
mean,557.7054841028381,1762.3493297644493,531.9380285562279,417.7393549347254,32692.86531098302,4744.425143230917,437.61265870903975
stddev,883.5580323607842,2792.043382260077,559.682244268939,79.05153265166982,9809.328308716891,2985.1908261919334,659.0336847244166
min,0.453592,1.43335072,1.61,15.863636363636363,-399.8976,0.0,0
max,32205.032000000003,101767.90112000001,3599.993,975.1272965879265,43006.0,13582.587877772714,7011


fuel_quantiles_kg
"List(16.329312, 99.99999999999432, 200.0, 598.74144, 4462.891688)"


## 4. Typ samolotu i proste porównania
Zliczamy liczbę przedziałów, a nie liczbę samolotów. Nie utożsamiaj liczby przedziałów z rozmiarem floty.

In [0]:
aircraft_col = "flight_aircraft_type" if "flight_aircraft_type" in train.columns else "aircraft_type"
assert aircraft_col in train.columns, "Nie znaleziono kolumny typu samolotu."
by_aircraft = (train.groupBy(aircraft_col)
    .agg(F.count("*").alias("segments"),
         F.countDistinct("flight_id").alias("flights"),
         F.avg("fuel_kg").alias("avg_fuel_kg"),
         F.avg("dur_s").alias("avg_duration_s"))
    .orderBy(F.desc("segments")))
show(by_aircraft, 40)

flight_aircraft_type,segments,flights,avg_fuel_kg,avg_duration_s
A20N,40488,3220,146.73321109743122,415.7061693588217
A320,31980,3023,289.81064411569713,425.70388017510834
A359,23942,1571,1272.0276279966597,791.6958318018553
B788,7469,583,887.782049859419,699.1052545186774
B738,7457,551,295.19394987743044,443.10070926646097
A332,5441,532,826.9564772443308,505.33570189303447
B789,3173,224,1035.808862046014,671.3125899779388
A21N,3138,393,217.76740502485669,389.90572848948375
A321,2572,285,277.8034291346812,414.63460692068395
B77W,1762,183,2646.8344312281492,1141.4558110102157


## 5. Zależności liniowe wybranych cech z paliwem
Korelacje mają charakter opisowy; różna długość przedziałów może zaburzać porównania.

In [0]:
from pyspark.sql.types import NumericType
candidates = ["dur_s", "route_km", "alt_mean", "alt_std", "gs_mean", "vr_mean", "mach_mean", "tas_mean", "cas_mean", "n_points"]
rows=[]
for c in candidates:
    if c in train.columns and isinstance(train.schema[c].dataType, NumericType):
        subset = train.select(F.col("fuel_kg"), F.col(c).cast("double").alias(c)).na.drop()
        try:
            corr = subset.stat.corr("fuel_kg", c)
        except Exception:
            corr = None
        rows.append((c, float(corr) if corr is not None else None))
show(spark.createDataFrame(rows, ["feature", "pearson_corr"]).orderBy(F.desc(F.abs("pearson_corr")))) if rows else print("Brak cech do analizy")

feature,pearson_corr
dur_s,0.7894187376820118
route_km,0.5159559368260491
tas_mean,0.36051563651316526
alt_mean,0.18815801818080574
gs_mean,0.17126812529902224
alt_std,0.12266400520575386
vr_mean,0.10509011231695786
n_points,0.10344537558857723
mach_mean,0.062088348330940686
cas_mean,0.022757365570237154



## 6. Wnioski z eksploracyjnej analizy danych (EDA)

### 6.1. Jakość i kompletność danych

Zbiór treningowy zawiera **131 530 przedziałów zużycia paliwa oraz 75 kolumn**.

Przeprowadzona kontrola wykazała:
- Brak zduplikowanych identyfikatorów `idx`.
- Brak brakujących identyfikatorów `idx` i `flight_id`.
- Brak wartości NULL i NaN w `fuel_kg`.
- Brak ujemnych wartości zużycia paliwa.

Oznacza to, że podstawowa zmienna docelowa jest kompletna, a identyfikatory przedziałów są unikalne.

### 6.2. Braki danych

Największe braki występują w parametrach pochodzących z pomiarów ACARS:

| Zmienna | Brakujące wartości |
|---|---:|
| `mach_std`, `tas_std`, `cas_std` | 100% |
| `cas_mean` | 83,77% |
| `tas_mean` | 66,20% |
| `mach_mean` | 35,97% |
| `gs_mean` | 35,08% |
| `alt_mean` | 3,17% |

Brak odchyleń standardowych Mach, TAS i CAS wynika z niewystarczającej liczby dostępnych pomiarów w poszczególnych przedziałach.

Wysoki odsetek braków w danych ACARS oznacza, że nie wszystkie parametry mogą być wykorzystane bez dodatkowego przetwarzania.

**Wnioski dla modelowania:**
- Usunąć z listy predyktorów kolumny zawierające 100% braków.
- Rozważyć imputację lub wykorzystanie wskaźników braków dla Mach, TAS i CAS.
- Sprawdzić przyczyny braków `groundspeed` i `vertical_rate`.
- Zachować pełne dane źródłowe bez usuwania kolumn.

### 6.3. Rozkład zużycia paliwa

Podstawowe statystyki `fuel_kg`:

| Statystyka | Zużycie paliwa |
|---|---:|
| Średnia | 557,71 kg |
| Mediana | 200,00 kg |
| Odchylenie standardowe | 883,56 kg |
| Minimum | 0,45 kg |
| Maksimum | 32 205,03 kg |
| 25. percentyl | 100,00 kg |
| 75. percentyl | 598,74 kg |
| 99. percentyl | 4 462,89 kg |

Rozkład zużycia paliwa jest silnie prawostronnie asymetryczny. Średnia jest znacznie wyższa od mediany, co wskazuje na obecność stosunkowo niewielkiej liczby przedziałów o bardzo dużym zużyciu paliwa.

Najwyższe wartości mogą wynikać m.in. z obsługi dużych samolotów, dłuższych przedziałów lub potencjalnych anomalii pomiarowych.

**Wniosek:** przed modelowaniem należy przeanalizować obserwacje skrajne. Nie należy usuwać ich automatycznie, ponieważ mogą odpowiadać rzeczywistym warunkom lotu.

### 6.4. Czas trwania przedziałów

Średni czas trwania przedziału wynosi **531,94 sekundy (około 8,87 minuty)**.

Zaobserwowano:
- Minimum: 1,61 sekundy.
- Maksimum: 3599,99 sekundy (około 60 minut).

Minimalny czas jest niezgodny z deklarowanym zakresem 5–60 minut.

**Wniosek:** konieczna jest dodatkowa kontrola przedziałów krótszych niż 300 sekund oraz sprawdzenie sposobu wyznaczania granic czasowych.

### 6.5. Porównanie typów samolotów

Najliczniej reprezentowane typy samolotów to:

| Typ | Liczba przedziałów | Średnie paliwo na przedział |
|---|---:|---:|
| A20N | 40 488 | 146,73 kg |
| A320 | 31 980 | 289,81 kg |
| A359 | 23 942 | 1272,03 kg |
| B788 | 7469 | 887,78 kg |
| B738 | 7457 | 295,19 kg |

Najwięcej obserwacji pochodzi z samolotów A20N i A320.

Widoczne są znaczne różnice średniego zużycia paliwa między typami samolotów. Mogą one wynikać z różnic konstrukcyjnych, masy, długości analizowanych przedziałów i charakterystyki wykonywanych lotów.

Przykładowo A20N ma niższe średnie zużycie paliwa na przedział niż A320, ale nie jest to jeszcze dowód wyższej efektywności paliwowej.

**Wniosek:** porównania należy przeprowadzać po uwzględnieniu czasu trwania przedziału, dystansu i warunków lotu. Konieczne jest również uwzględnienie nierównej liczebności poszczególnych typów.

### 6.6. Korelacje ze zużyciem paliwa

Obliczono współczynnik korelacji Pearsona pomiędzy `fuel_kg` a wybranymi cechami.

| Zmienna | Korelacja z `fuel_kg` |
|---|---:|
| `dur_s` | 0,789 |
| `route_km` | 0,516 |
| `tas_mean` | 0,361 |
| `alt_mean` | 0,188 |
| `gs_mean` | 0,171 |
| `vr_mean` | 0,105 |
| `mach_mean` | 0,062 |
| `cas_mean` | 0,023 |

Najsilniejszą dodatnią zależność liniową ze zużyciem paliwa wykazuje **czas trwania przedziału (`dur_s`)**.

Umiarkowana dodatnia korelacja występuje również dla dystansu między lotniskami (`route_km`).

Pozostałe zmienne wykazują słabsze korelacje liniowe, co nie oznacza, że są nieprzydatne. Ich wpływ może być nieliniowy lub zależny od typu samolotu i fazy lotu.

**Wniosek:** czas trwania przedziału i dystans trasy są ważnymi kandydatami na predyktory. W modelowaniu warto zastosować również algorytmy uwzględniające zależności nieliniowe.

### 6.7. Wnioski do dalszego modelowania

Na podstawie EDA przyjęto następujące kierunki dalszej pracy:

1. Zachowanie `fuel_kg` jako zmiennej docelowej.
2. Wykluczenie `co2_kg` z predyktorów, ponieważ jest bezpośrednio obliczane z `fuel_kg`.
3. Sprawdzenie i obsługa braków danych w parametrach lotu.
4. Kontrola nietypowo krótkich przedziałów i skrajnych wartości zużycia paliwa.
5. Uwzględnienie typu samolotu, długości przedziału, parametrów lotu i dystansu trasy.
6. Porównanie modeli liniowych i nieliniowych.
7. Ocena modeli na oddzielnym zbiorze walidacyjnym oraz końcowa ocena na zbiorze testowym.

### 6.8. Podsumowanie

Eksploracyjna analiza danych wykazała, że zbiór treningowy jest kompletny pod względem identyfikatorów i zmiennej docelowej, jednak występują istotne braki w części parametrów trajektorii.

Zużycie paliwa cechuje się dużą zmiennością i prawostronną asymetrią. Najsilniejszą obserwowaną zależność liniową wykazuje czas trwania przedziału, a istotnym kandydatem na predyktor jest również dystans trasy.

Porównanie typów samolotów wskazuje na różnice w średnim zużyciu paliwa, jednak ich właściwa interpretacja wymaga uwzględnienia długości przedziałów i warunków lotu.

**Dane nadają się do dalszego modelowania po przeprowadzeniu kontroli anomalii, analizy braków oraz odpowiedniego przygotowania cech.**

Na podstawie obecnego zbioru nie można wyciągać bezpośrednich wniosków o efektywności konkretnych linii lotniczych ani o rzeczywistym wieku poszczególnych samolotów.
